In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install diffusers accelerate transformers datasets safetensors torch torchvision peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 31.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [3]:
!pip install bitsandbytes
!pip install --upgrade transformers diffusers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 64.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.0
    Uninstalling transformers-4.53.0:
      Successfully uninstalled transformers-4.53.0


In [4]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import json

In [5]:
from huggingface_hub import model_info
from diffusers import StableDiffusionPipeline
from diffusers import StableDiffusionPipeline
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch

In [6]:
from torch.utils.data import Dataset, DataLoader
import os
import json
from PIL import Image
from torchvision import transforms
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL

In [18]:
json_path = "/content/drive/MyDrive/flickr8k/flickr8k_captions.json"
image_folder = "/content/drive/MyDrive/flickr8k/Flickr8k_Dataset/Images"
output_json = "/content/drive/MyDrive/flickr8k/flickr8k_captions.json"

In [19]:
with open(json_path, "r") as f:
    raw_data = json.load(f)

# Filter out entries with missing images
dataset = [item for item in raw_data if os.path.exists(os.path.join(image_folder, item["image_sticker_name"]))]

# Optional: limit for quick testing
# dataset = dataset[:100]

print(f"✅ Filtered dataset loaded: {len(dataset)} entries")

✅ Filtered dataset loaded: 8091 entries


In [20]:
import pandas as pd
import json

# Load CSV
df = pd.read_csv(captions_path)

# Keep only one caption per image (e.g., the first one)
df_unique = df.groupby("image").first().reset_index()

# Convert to required JSON format
final_data = []
for _, row in df_unique.iterrows():
    entry = {
        "image_sticker_name": row["image"],
        "image_sticker_caption": row["caption"]
    }
    final_data.append(entry)

# Save to JSON
with open(output_json, "w") as f:
    json.dump(final_data, f, indent=2)

print(f"✅ Saved {len(final_data)} unique image-caption pairs to JSON.")


✅ Saved 8091 unique image-caption pairs to JSON.


In [21]:
# Load dataset
with open(output_json, "r") as f:
    dataset = json.load(f)
print(f"✅ Loaded {len(dataset)} caption-image pairs")

✅ Loaded 8091 caption-image pairs


In [22]:
# Load tokenizer, text encoder, VAE
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to("cuda")
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to("cuda")

In [23]:
# Define Custom Dataset
class StickerDataset(Dataset):
    def __init__(self, dataset, image_folder, tokenizer, text_encoder, vae, max_seq_length=50):
        self.dataset = dataset
        self.image_folder = image_folder
        self.tokenizer = tokenizer
        self.text_encoder = text_encoder
        self.vae = vae
        self.max_seq_length = max_seq_length
        self.image_transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]["image_sticker_caption"]
        tokens = self.tokenizer(text, padding="max_length", truncation=True, max_length=self.max_seq_length, return_tensors="pt").to("cuda")
        with torch.no_grad():
            text_embeddings = self.text_encoder(tokens.input_ids).last_hidden_state

        image_path = os.path.join(self.image_folder, self.dataset[idx]["image_sticker_name"])
        image = Image.open(image_path).convert("RGB")
        image_tensor = self.image_transform(image).unsqueeze(0).to("cuda")

        with torch.no_grad():
            latents = self.vae.encode(image_tensor).latent_dist.sample().squeeze(0) * 0.18215
            latents = latents.cpu()

        timestep = torch.randint(0, 1000, (1,))
        return {
            "sample": latents,
            "timestep": timestep,
            "encoder_hidden_states": text_embeddings.squeeze(0).cpu()
        }

In [24]:
# Create DataLoader
Sticker_dataset = StickerDataset(dataset, image_folder, tokenizer, text_encoder, vae)
train_dataloader = DataLoader(Sticker_dataset, batch_size=4, shuffle=True, pin_memory=False)
print(f"✅ DataLoader ready with {len(train_dataloader)} batches")

✅ DataLoader ready with 2023 batches


In [25]:
model_name = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to("cuda")


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [26]:
# Enable LoRA on UNet and text encoder
lora_config_clip = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none"
)
pipe.text_encoder = get_peft_model(pipe.text_encoder, lora_config_clip)

lora_config_unet = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["attn1.to_q", "attn1.to_k", "attn1.to_v", "attn2.to_out.0"],
    lora_dropout=0.1,
    bias="none"
)
pipe.unet = get_peft_model(pipe.unet, lora_config_unet)

print("✅ LoRA injected into UNet and CLIP Text Encoder")

✅ LoRA injected into UNet and CLIP Text Encoder


In [27]:
from tqdm import tqdm
import torch.nn.functional as F
# Fine-tuning loop
pipe.unet.train().to("cuda")
optimizer = torch.optim.AdamW(pipe.unet.parameters(), lr=2e-6)
num_epochs = 5  # Start with 5 for testing; increase later

for epoch in range(num_epochs):
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch in progress_bar:
        optimizer.zero_grad()
        latents = batch["sample"].to("cuda").to(torch.float16)
        timesteps = batch["timestep"].squeeze(-1).to("cuda")
        encoder_hidden_states = batch["encoder_hidden_states"].to("cuda").to(torch.float16)
        noise = torch.randn_like(latents, dtype=torch.float16)
        noisy_latents = latents + noise * 0.1
        predicted_noise = pipe.unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(predicted_noise, noise)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix({"Loss": loss.item()})

    print(f"✅ Epoch {epoch+1} completed with average loss: {total_loss / len(train_dataloader):.4f}")

print("🎉 Fine-tuning complete!")


Epoch 1/5: 100%|██████████| 2023/2023 [1:13:05<00:00,  2.17s/it, Loss=0.961]


✅ Epoch 1 completed with average loss: 1.0244


Epoch 2/5: 100%|██████████| 2023/2023 [07:55<00:00,  4.25it/s, Loss=0.947]


✅ Epoch 2 completed with average loss: 0.9761


Epoch 3/5: 100%|██████████| 2023/2023 [07:55<00:00,  4.26it/s, Loss=0.933]


✅ Epoch 3 completed with average loss: 0.9617


Epoch 4/5: 100%|██████████| 2023/2023 [07:56<00:00,  4.25it/s, Loss=0.956]


✅ Epoch 4 completed with average loss: 0.9532


Epoch 5/5: 100%|██████████| 2023/2023 [07:56<00:00,  4.25it/s, Loss=0.968]

✅ Epoch 5 completed with average loss: 0.9498
🎉 Fine-tuning complete!


In [28]:
from peft import get_peft_model_state_dict

# Create output directory
output_dir = "/content/drive/MyDrive/fine_tuned_sticker_model"
os.makedirs(output_dir, exist_ok=True)

# Save LoRA adapters
torch.save(get_peft_model_state_dict(pipe.unet), os.path.join(output_dir, "unet_lora.pth"))
torch.save(get_peft_model_state_dict(pipe.text_encoder), os.path.join(output_dir, "text_encoder_lora.pth"))

print("✅ LoRA adapters saved to Google Drive.")


✅ LoRA adapters saved to Google Drive.


In [29]:
from PIL import Image
import torch

pipe.unet.eval()  # Set model to inference mode

prompt = "a cute cartoon sticker of a black dog with sunglasses"

# Generate image
with torch.autocast("cuda"):
    image = pipe(prompt).images[0]

# ✅ Show the image inside Colab
image.show()

# ✅ Save the image so you can download
image.save("dog_sticker.png")

  0%|          | 0/50 [00:00<?, ?it/s]

In [30]:
!pip install fastapi uvicorn python-multipart nest-asyncio pyngrok

In [32]:
!ngrok config add-authtoken 2y5cxjzprgRfHNubfgxPlQB9d2P_7wa4Ymqv6oWzCP21UYEVb

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [34]:
from fastapi import FastAPI, Form
from fastapi.responses import FileResponse
import uvicorn, nest_asyncio
from pyngrok import ngrok
import uuid
import torch

# Make sure model is loaded as `pipe` and is on .eval() mode

app = FastAPI()
pipe.unet.eval()
pipe.text_encoder.eval()

@app.post("/generate-sticker/")
def generate_sticker(prompt: str = Form(...)):
    with torch.autocast("cuda"):
        image = pipe(prompt).images[0]
    filename = f"{uuid.uuid4().hex}.png"
    image.save(filename)
    return FileResponse(filename, media_type="image/png")

# 👉 Set up FastAPI in Colab
nest_asyncio.apply()
ngrok_tunnel = ngrok.connect(8000)
print("🚀 Public URL:", ngrok_tunnel.public_url)

# ✅ Start FastAPI server (run this last, after all above is ready)
uvicorn.run(app, host="0.0.0.0", port=8000)



🚀 Public URL: https://91162ec4fd07.ngrok-free.app


INFO:     Started server process [877]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     204.101.131.2:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     204.101.131.2:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [877]
